# Make figures for manuscript

## Setup

In [ ]:
import os
import sys
import re
import string
from glob import glob
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray as rxr
import xarray as xr
import scores
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import linregress
from metloom.pointdata import SnotelPointData

sys.path.append('../scripts')
from model_comparison_functions import (
    get_model_prep_outputs, x_y_snotel
)

# Root directory containing the per-date "dates/<YYYYMMDD>/" folders produced by Model_prep.py.
# Model_prep.py outputs are loaded by exact filename with get_model_prep_outputs(), using the
# dataset names "HMS-EB", "HMS-TI", "iSnobal", "SnowModel", "LiDAR", and "LiDAR-2000".
base_dir = "/Users/rdcrlrka/Research/SIRO/SIRO_P1/MCSModeling/dates"

# Where to save figures
fig_dir = "/Users/rdcrlrka/Research/SIRO/SIRO_P1/MCSModeling/figures"
os.makedirs(fig_dir, exist_ok=True)

# Model_prep.py outputs for the baseline (Task 1) and assimilation (Task 2) runs, loaded once
task1 = get_model_prep_outputs(base_dir, 1)
task2 = get_model_prep_outputs(base_dir, 2)

# Plot settings
plt.rcParams.update({'font.size': 12, "font.sans-serif": "Verdana"})

# Models, in plotting order, and their display names
MODEL_NAMES = ["HMS-EB", "HMS-TI", "iSnobal", "SnowModel"]
MODEL_NAMES_DISPLAY = {
    "HMS-EB": "HEC-HMS EB",
    "HMS-TI": "HEC-HMS TI",
    "iSnobal": "iSnobal",
    "SnowModel": "SnowModel",
}

# Panel labels and grid lines, shared by all figures
LABEL_BOX = dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1)
GRID_STYLE = dict(color='k', linewidth=0.4, alpha=0.4)

def add_panel_label(ax, label, loc='upper left', fontsize=14):
    # Determine xy and alignment based on location 
    # so there is always a 0.5 x 0.5 pad with the axes
    if loc=='upper left':
        x,y = 0.05, 0.95
        ha, va = 'left', 'top'
    elif loc=='upper right':
        x,y = 0.95, 0.95
        ha, va = 'right', 'top'
    elif loc=='lower left':
        x,y = 0.05, 0.05
        ha, va = 'left', 'bottom'
    elif loc=='lower right':
        x,y = 0.95, 0.05
        ha, va = 'right', 'bottom'

    ax.text(
        x, y, label, transform=ax.transAxes, ha=ha, va=va, 
        fontweight='bold', bbox=LABEL_BOX, fontsize=fontsize
        )

## Define colormaps

In [ ]:
# Snow depth
sd_cmap = plt.cm.viridis
sd_cmap

In [ ]:
# Datasets
ds_colors_dict = {
    "SPIReS":       "#000000",
    "LiDAR-100 m":        "#8E8E8E",
    "LiDAR-2,000 m":   "#CECECE",
    "HEC-HMS EB":       "#7D2B54",
    "HEC-HMS TI":       "#BF6B78",
    "iSnobal":      "#97CAEA",
    "SnowModel":    "#37753B"
}
ds_colors = list(ds_colors_dict.values())
ds_cmap = matplotlib.colors.ListedColormap(ds_colors)
ds_cmap

In [ ]:
base_run_color = '#1b9e77'
assim_run_color = '#d95f02'

base_run_cmap = matplotlib.colors.LinearSegmentedColormap.from_list(name='baseline_cmap', colors=['w', base_run_color])
assim_run_cmap = matplotlib.colors.LinearSegmentedColormap.from_list(name='assimilation_cmap', colors=['w', assim_run_color])

assim_run_cmap

## Example snow depths map view + basin outline

In [ ]:
# Load required input files
example_date = "20230405"

mcs_shp = gpd.read_file(os.path.join(base_dir, example_date, "MCS_outline", "MCS_outline.shp"))
basin_shp = gpd.read_file(os.path.join(base_dir, example_date, "MCS_outline", "basin_outline.shp"))

basin_rasters = task1[example_date]['basin_clip']
raster_list = {m: basin_rasters[m] for m in MODEL_NAMES}
raster_list

In [ ]:
# Make the figure and save
fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
axes = axes.flatten()
km_formatter = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v / 1e3:.0f}")

for i, (model, raster_path) in enumerate(raster_list.items()):
    with rxr.open_rasterio(raster_path, mask_and_scale=True).squeeze() as raster:
        im = axes[i].imshow(
            raster.data, cmap=sd_cmap,  vmin=0, vmax=4.0,
            extent=(min(raster.x), max(raster.x), min(raster.y), max(raster.y)),
            )

    basin_shp.plot(ax=axes[i], facecolor="none", edgecolor="k", linewidth=1.5)
    mcs_shp.plot(ax=axes[i], facecolor="none", edgecolor="orange", linewidth=1.5)

    axes[i].set_title(MODEL_NAMES_DISPLAY[model])
    add_panel_label(axes[i], string.ascii_lowercase[i], loc='upper left')
    axes[i].grid(False)
    axes[i].set_aspect("equal")

# Axes in km
for ax in axes:
    ax.set_xlim(570e3, 610e3)
    ax.set_ylim(4830e3, 4880e3)
    ax.xaxis.set_major_formatter(km_formatter)
    ax.yaxis.set_major_formatter(km_formatter)
for ax in [axes[0], axes[2]]:
    ax.set_ylabel("Northing (km)")
for ax in [axes[2], axes[3]]:
    ax.set_xlabel("Easting (km)")

# colorbar
cbar = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label("Snow Depth (m)", fontsize=12)
cbar.ax.tick_params(labelsize=10)

plt.show()

fig_file = os.path.join(fig_dir, f"map_view_snow_depth_{example_date}.png")
fig.savefig(fig_file, dpi=300, bbox_inches="tight")
print("Figure saved to:", fig_file)

## SNOTEL and LiDAR dates

In [ ]:
# Load required input files
snotel_point_MCS = SnotelPointData("637:ID:SNTL", "MCS")

MCS_SNOTEL = snotel_point_MCS.get_daily_data(
    datetime(2022, 10, 1), datetime(2025, 10, 1),
    [snotel_point_MCS.ALLOWED_VARIABLES.SNOWDEPTH]
)
MCS_SNOTEL["SNOWDEPTH (m)"] = MCS_SNOTEL["SNOWDEPTH"] * 0.0254  # inches -> meters

# LiDAR flight dates: which were used for data assimilation vs. held out for validation
validation_dates = pd.to_datetime(['2023-04-05', '2024-03-15', '2024-04-18', '2025-04-04', '2025-05-01'])
assimilation_dates = pd.to_datetime(['2022-12-08', '2023-02-09', '2023-03-16', '2023-12-28',
                                      '2024-01-15', '2024-02-13', '2025-01-13', '2025-01-29'])

# 30-year daily-mean for context
MCS_historic = snotel_point_MCS.get_daily_data(
    datetime(1995, 10, 1), datetime(2026, 6, 1),
    [snotel_point_MCS.ALLOWED_VARIABLES.SNOWDEPTH]
).reset_index(level=[1])
MCS_historic['SNOWDEPTH (m)'] = MCS_historic['SNOWDEPTH'] * 0.0254
MCS_historic.reset_index(inplace=True)
MCS_historic['datetime'] = pd.to_datetime(MCS_historic['datetime'])
MCS_historic['MM-DD'] = MCS_historic['datetime'].dt.strftime('%m-%d')
daily_avg = MCS_historic.groupby('MM-DD')['SNOWDEPTH (m)'].mean()

In [ ]:
# Make the figure and save
dates = MCS_SNOTEL.index.get_level_values('datetime')
plot_mmdd = pd.to_datetime(dates).strftime('%m-%d')
clim_snowdepth = plot_mmdd.map(daily_avg)

lw = 3

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(dates, MCS_SNOTEL['SNOWDEPTH (m)'], linewidth=lw, label='SNOTEL Depth', alpha=0.9, color='k')
ax.plot(dates, clim_snowdepth, color='black', linestyle='-', linewidth=lw, alpha=0.5, label='SNOTEL 30-Year Mean')

for i, d in enumerate(assimilation_dates):
    ax.axvline(x=d, color=assim_run_color, linestyle='dotted', linewidth=lw,
               label='Assimilation Flight' if i == 0 else "_nolegend_")

for i, d in enumerate(validation_dates):
    ax.axvline(x=d, color=base_run_color, linestyle='dashdot', linewidth=lw, alpha=0.7,
               label='Validation Flight' if i == 0 else "_nolegend_")

ax.set_xlabel('Date', fontsize=18)
ax.set_ylabel('Snow Depth (m)', fontsize=18)
ax.tick_params(axis='both', labelsize=18)
ax.legend(fontsize=18, framealpha=1, facecolor='white', ncol=2, loc='upper center')
ax.axhline(0, color='k', lw=lw/2)
ax.set_ylim(0, 2.9)

plt.tight_layout()
fig_file = os.path.join(fig_dir, "SNOTEL_timeseries_lidar_flights.png")
plt.savefig(fig_file, dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to:", fig_file)

## LiDAR comparison

### Snow depth distribution box plots

In [ ]:
# Load required input files
dfs = []
for date, contents in task1.items():
    if 'lidar_clip' not in contents:
        continue
    for model, raster_path in contents['lidar_clip'].items():
        with rasterio.open(raster_path) as src:
            data = src.read(1)
            data_masked = np.ma.array(data, mask=(data == -9999))
            flattened = data_masked.compressed()
            flattened = flattened[flattened < 5]

        dfs.append(pd.DataFrame({
            "Date": date,
            "Model": model,
            "Snow Depth": flattened,
        }))

lidar_domain = pd.concat(dfs, ignore_index=True)

# Rename the columns for plotting
lidar_domain.loc[lidar_domain['Model']=='LiDAR', 'Model'] = 'LiDAR-100 m'
lidar_domain.loc[lidar_domain['Model']=='LiDAR-2000', 'Model'] = 'LiDAR-2,000 m'
for model in MODEL_NAMES:
    lidar_domain.loc[lidar_domain['Model']==model, 'Model'] = MODEL_NAMES_DISPLAY[model]

lidar_domain

In [ ]:
# Make the figure and save
date_order = sorted(lidar_domain['Date'].unique())
model_order = ['LiDAR-100 m', 'LiDAR-2,000 m'] + [MODEL_NAMES_DISPLAY[m] for m in ["iSnobal", "SnowModel", "HMS-EB", "HMS-TI"]]
xtick_labels = [f"{d[0:4]}-{d[4:6]}-{d[6:]}" for d in date_order]

fig, ax = plt.subplots(figsize=(15, 8))

sns.boxplot(
    data=lidar_domain,
    x="Date", y="Snow Depth", hue="Model", palette=ds_colors_dict,
    order=date_order, hue_order=model_order, gap=0.1, ax=ax,
)

ax.set_xlabel('Date', fontsize=22)
ax.set_ylabel('Snow Depth (m)', fontsize=22)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.set_xticks(range(len(date_order)), xtick_labels)
ax.legend(prop={'size': 20}, ncol=3, loc='upper center')
ax.axhline(0, color='k', lw=1)

plt.tight_layout()
plt.show()

fig_file = os.path.join(fig_dir, "lidar_distributions_v2.png")
fig.savefig(fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", fig_file)

### Regression scatter plots

In [ ]:
date_order = sorted(task1.keys())

# Version 2, dates combined, hexbins for iSnobal and SnowModel
hexbin_models = {"iSnobal", "SnowModel"}
markersize = 20

task_rows = [
    ("Baseline Run", task1, base_run_color, base_run_cmap),
    ("Assimilation Run", task2, assim_run_color, assim_run_cmap),
]

fig, axs = plt.subplots(2, len(MODEL_NAMES), figsize=(12, 6))

for row, (task_label, task_dict, color, cmap) in enumerate(task_rows):
    for col, model in enumerate(MODEL_NAMES):
        ax = axs[row, col]

        # Pool all validation dates together
        x_all, y_all = [], []
        for date in date_order:
            lidar_path = task_dict[date]["lidar_2000"] if "HMS" in model else task_dict[date]["lidar"]
            lidar_flat = rxr.open_rasterio(lidar_path, masked=True).squeeze()
            lidar_flat = lidar_flat.where(lidar_flat < 5).values.flatten()

            modeled_flat = rxr.open_rasterio(task_dict[date]["resample"][model], masked=True).squeeze()
            modeled_flat = modeled_flat.values.flatten()

            mask = np.isfinite(lidar_flat) & np.isfinite(modeled_flat)
            x_all.append(lidar_flat[mask])
            y_all.append(modeled_flat[mask])

        x_all = np.concatenate(x_all)
        y_all = np.concatenate(y_all)

        if model in hexbin_models:
            hb = ax.hexbin(x_all, y_all, gridsize=20, cmap=cmap, mincnt=1, extent=(0, 4, 0, 4), clim=(0,300))
            if col==3:  
                fig.colorbar(hb, ax=ax, fraction=0.05, pad=0.04, label="Count")
        else:
            ax.scatter(x_all, y_all, s=markersize, alpha=1, color=color)

        ax.plot([0, 4], [0, 4], color='k', linestyle='--')
        ax.set_xlim(0, 4)
        ax.set_ylim(0, 4)

        if row == 1:
            ax.set_xlabel("LiDAR Snow Depth (m)")#, fontweight='bold')
        else:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)

        if col == 0:
            ax.set_ylabel(f"{task_label}\nModeled Snow Depth (m)")
        else:
            ax.set_ylabel("")

        add_panel_label(
            ax, string.ascii_lowercase[row * len(MODEL_NAMES) + col],
            loc='upper left'
            )

        # Add R2 annotation
        # reg = linregress(x_all, y_all)
        # r2 = reg.rvalue**2
        # ax.text(0.95, 0.05, f"R$^2$ = {r2:.2f}", transform=ax.transAxes, va='bottom', ha='right')

        # Model name
        if row==0:
            ax.set_title(MODEL_NAMES_DISPLAY[model])


fig.tight_layout()
plt.show()

fig_file = os.path.join(fig_dir, "modeled_vs_lidar_scatter.png")
fig.savefig(fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", fig_file)

### Data tables

In [ ]:
def rmse_numpy(actual, predicted):
    return np.sqrt(np.mean((predicted - actual) ** 2))

regressions = []
for date in task1.keys():
    lidar_100 = rxr.open_rasterio(task1[date]['lidar'], masked=True).squeeze()
    lidar_100 = lidar_100.where(lidar_100 < 5)
    lidar_100_flat = lidar_100.values.flatten()

    lidar_2000 = rxr.open_rasterio(task1[date]['lidar_2000'], masked=True).squeeze()
    lidar_2000 = lidar_2000.where(lidar_2000 < 5)
    lidar_2000_flat = lidar_2000.values.flatten()

    resample_tasks = {"Task 1": task1[date]['resample'], "Task 2": task2[date]['resample']}

    for task_name, resampled_models in resample_tasks.items():
        for model_name, model_path in resampled_models.items():
            modeled = rxr.open_rasterio(model_path, masked=True).squeeze()
            modeled_flat = modeled.values.flatten()

            lidar_flat = lidar_2000_flat if "HMS" in model_name else lidar_100_flat

            mask = ~np.isnan(lidar_flat) & ~np.isnan(modeled_flat)
            x_clean = lidar_flat[mask]
            y_clean = modeled_flat[mask]

            if len(x_clean) > 1 and len(y_clean) > 1:
                reg = linregress(x_clean, y_clean)
                regressions.append({
                    "Task": task_name,
                    "Condition": "Assimilation" if task_name == "Task 2" else "Baseline",
                    "Date": date,
                    "Model": model_name,
                    "slope": reg.slope,
                    "intercept": reg.intercept,
                    "rvalue": reg.rvalue,
                    "pvalue": reg.pvalue,
                    "stderr": reg.stderr,
                    "RMSE": rmse_numpy(x_clean, y_clean),
                })

regression_df = pd.DataFrame(regressions)
regression_df['r-squared'] = regression_df['rvalue'] ** 2


# Change model names for display
for model in MODEL_NAMES:
    regression_df.loc[regression_df['Model']==model, 'Model'] = MODEL_NAMES_DISPLAY[model]

# Restructure to match the format in the paper
regression_df['Date'] = pd.to_datetime(regression_df['Date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
pivoted = regression_df.pivot(
    index=['Date', 'Model'], 
    columns='Condition', 
    values=['r-squared', 'RMSE', 'pvalue']
)
pivoted = pivoted.swaplevel(axis=1)
pivoted = pivoted[['Baseline', 'Assimilation']]
pivoted = pivoted.reindex(['r-squared', 'RMSE', 'pvalue'], level=1, axis=1)
pivoted = pivoted.reindex([MODEL_NAMES_DISPLAY[m] for m in MODEL_NAMES], level='Model')  # same model order as the figures

regression_file = os.path.join(fig_dir, "lidar_regression_results.csv")
pivoted.to_csv(regression_file, index=True)
print("Saved regression parameters to:", regression_file)
pivoted.round(2)

In [ ]:
# Print in LaTeX form for the paper
cols = [(run, stat) for run in ['Baseline', 'Assimilation'] for stat in ['r-squared', 'RMSE', 'pvalue']]

lines = [
    r"\begin{tabular}{ll ccc ccc}",
    r"\toprule",
    r" & & \multicolumn{3}{c}{Baseline Run} & \multicolumn{3}{c}{Assimilation Run} \\",
    r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
    r"Date & Model & $R^2$ & RMSE (m) & $p$-value & $R^2$ & RMSE (m) & $p$-value \\",
    r"\midrule",
]
groups = list(pivoted[cols].groupby(level='Date', sort=False))
for g, (date, group) in enumerate(groups):
    for i, ((_, model), row) in enumerate(group.iterrows()):
        first = rf"\multirow{{{len(group)}}}{{*}}{{{date}}}" if i == 0 else ""
        values = " & ".join(f"{v:.2f}" for v in row)
        lines.append(f"{first} & {model} & {values} \\\\")
    if g < len(groups) - 1:
        lines.append(r"\cline{1-8}")
lines += [r"\bottomrule", r"\end{tabular}"]

print("\n".join(lines))

### R$^2$ and RMSE time series

In [ ]:
model_display_order = [MODEL_NAMES_DISPLAY[m] for m in MODEL_NAMES]
plot_df = regression_df.sort_values('Date')   # categorical x axis follows row order, so sort by date

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

for axis, metric, ylabel in zip(ax, ['r-squared', 'RMSE'], ["R$^2$", "RMSE (m)"]):
    sns.lineplot(
        ax=axis,
        data=plot_df,
        x='Date',
        y=metric,
        hue='Model',
        hue_order=model_display_order,
        palette=ds_colors_dict,
        style='Condition',
        style_order=['Baseline', 'Assimilation'],
        markers=True,
        markersize=8,
    )
    axis.set_xlabel("Validation Date")
    axis.set_ylabel(ylabel)

# One shared legend, built by name (not by sort position)
handles, labels = ax[0].get_legend_handles_labels()
lookup = dict(zip(labels, handles))
for axis in ax:
    axis.get_legend().remove()
fig.legend([lookup[m] for m in model_display_order], model_display_order,
           loc='upper center', frameon=False, ncol=4, bbox_to_anchor=[0.4, 0.9, 0.2, 0.2])
fig.legend([lookup[c] for c in ['Baseline', 'Assimilation']], ['Baseline Run', 'Assimilation Run'],
           loc='upper center', frameon=False, ncol=2, bbox_to_anchor=[0.4, 0.85, 0.2, 0.2])

for label, axis in zip(string.ascii_lowercase, ax):
    add_panel_label(axis, label, loc='lower right')
    axis.set_ylim(0, 1)
    axis.grid(which='both', **GRID_STYLE)
    axis.tick_params(axis='both', which='major', labelsize=10)

plt.tight_layout()
plt.show()

fig_file = os.path.join(fig_dir, "lidar_r2_rmse.png")
fig.savefig(fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", fig_file)

## SNOTEL comparison

In [ ]:
# --- Load daily modeled + observed snow depth at each SNOTEL station ---
model_outputs_dir = "/Users/rdcrlrka/Research/SIRO/SIRO_P1/MCSModeling/ModelOutputs"
station_list = [
    ("637", "Mores Creek Summit"),
    ("978", "Bogus Basin"),
]
INCHES_TO_M = 0.0254
# Must match the run names in the cached CSVs and the plotting cells below
task_labels = {1: "Baseline", 2: "Assimilation"}
EXPECTED_MODELS = set(MODEL_NAMES)

if not os.path.isdir(model_outputs_dir):
    raise FileNotFoundError(f"Model outputs folder not found: {model_outputs_dir}")

def build_daily_samples(station_id, station_name):
    pt = SnotelPointData(f"{station_id}:ID:SNTL", station_name)
    sx, sy = x_y_snotel(pt)
    records = []  # (date, model, task, snow_depth_m)

    def sample_tif(path, scale=1.0):
        with rasterio.open(path) as src:
            row, col = src.index(sx, sy)
            val = src.read(1, window=((row, row + 1), (col, col + 1)))[0, 0]
            return np.nan if val == src.nodata else val * scale

    # --- SNOTEL (observed, inches) ---
    SNOTEL = pt.get_daily_data(
        datetime(2022, 10, 1), datetime(2025, 10, 1), [pt.ALLOWED_VARIABLES.SNOWDEPTH]
    )
    snotel_series = SNOTEL['SNOWDEPTH'].reset_index(level='site', drop=True) * INCHES_TO_M
    # metloom returns tz-aware 08:00 UTC stamps; strip so dates line up with the model dates
    snotel_series.index = pd.to_datetime(snotel_series.index).tz_localize(None).normalize()
    for date, val in snotel_series.items():
        records.append((date, "SNOTEL", "Observed", val))

    # --- HMS (daily geotiffs, inches) ---
    hms_daily_dirs = {
        ("HMS-EB", 1): os.path.join(model_outputs_dir, "HMS_Task1", "EB", "snow_depth_tif"),
        ("HMS-EB", 2): os.path.join(model_outputs_dir, "HMS_Task2", "EB", "snow_depth_tif"),
        ("HMS-TI", 1): os.path.join(model_outputs_dir, "HMS_Task1", "TI", "snow_depth_tif"),
        ("HMS-TI", 2): os.path.join(model_outputs_dir, "HMS_Task2", "TI", "snow_depth_tif"),
    }
    hms_date_pattern = re.compile(r"(\d{4}_\d{2}_\d{2})T0000")
    for (model_name, task), tif_dir in hms_daily_dirs.items():
        for path in sorted(glob(os.path.join(tif_dir, "*.tif*"))):
            m = hms_date_pattern.search(os.path.basename(path))
            if m:
                date = pd.to_datetime(m.group(1), format="%Y_%m_%d")
                records.append((date, model_name, task_labels[task], sample_tif(path, INCHES_TO_M)))

    # --- iSnobal (daily geotiffs, meters) ---
    isnobal_dirs = {
        1: os.path.join(model_outputs_dir, "m3w_isnobal_task1", "m3w_isnobal_task1_depth"),
        2: os.path.join(model_outputs_dir, "m3w_isnobal_task_2_all_updated", "m3w_isnobal_task2_depth"),
    }
    isnobal_date_pattern = re.compile(r"depth_(\d{4}-\d{2}-\d{2})T")
    for task, base in isnobal_dirs.items():
        for path in sorted(glob(os.path.join(base, "wy*", "*.tif"))):
            m = isnobal_date_pattern.search(os.path.basename(path))
            if m:
                date = pd.to_datetime(m.group(1), format="%Y-%m-%d")
                records.append((date, "iSnobal", task_labels[task], sample_tif(path)))

    # --- SnowModel (netCDF per water year, meters) ---
    for task in (1, 2):
        for nc_path in sorted(glob(os.path.join(model_outputs_dir, "SnowModel", f"SnowModel_WY*_Task{task}.nc"))):
            with xr.open_dataset(nc_path) as ds:
                series = ds["SD"].sel(x=sx, y=sy, method="nearest").to_series()
            for date, val in series.items():
                records.append((pd.Timestamp(date).normalize(), "SnowModel", task_labels[task], val))

    df = pd.DataFrame.from_records(records, columns=["date", "model", "task", "snow_depth_m"])

    # Fail loudly instead of silently returning SNOTEL-only data (e.g., if a path is wrong)
    missing = EXPECTED_MODELS - set(df["model"])
    if missing:
        raise RuntimeError(f"{station_name}: no model files found for {sorted(missing)} under {model_outputs_dir}")

    # unstack (unlike pivot_table) raises on duplicate dates instead of silently averaging them
    return df.set_index(["date", "model", "task"])["snow_depth_m"].unstack(["model", "task"])

snotel_dict = {}
for station_id, station_name in station_list:
    csv_path = os.path.join(fig_dir, f"SNOTEL_model_daily_samples_{station_name.replace(' ', '')}.csv")
    df = None
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, header=[0, 1], index_col=0, parse_dates=True)
        cached_models = set(df.columns.get_level_values(0))
        if EXPECTED_MODELS <= cached_models:
            print(f"Daily SNOTEL samples already exist for {station_name}, loading.")
        else:
            print(f"Cached samples for {station_name} are missing {sorted(EXPECTED_MODELS - cached_models)}; rebuilding.")
            df = None
    if df is None:
        print(f"Building daily samples for {station_name}...")
        df = build_daily_samples(station_id, station_name)
        df.to_csv(csv_path)
        print(f"Saved {len(df)} days to {csv_path}")
    # Only drop dates with no data at all; per-model gaps are handled when plotting
    df = df.dropna(how="all")
    snotel_dict[station_name] = {"df": df}

# Shared by the SNOTEL figures below
station_order = [name for _, name in station_list]
run_styles = [(task_labels[1], base_run_color), (task_labels[2], assim_run_color)]

snotel_dict["Mores Creek Summit"]["df"]

In [ ]:
lim = (0, 3)
lw = 1.5
ms = 30
alpha = 0.6
sd_ticks = [0, 1, 2, 3]

n_rows = 2 * len(station_order)
panel_labels = np.array(list(string.ascii_lowercase[:n_rows * len(MODEL_NAMES)])).reshape(n_rows, len(MODEL_NAMES))

# One subfigure per station, with a gap between them for the station title + separator line
fig = plt.figure(figsize=(12, 13), layout='constrained')
subfigs = fig.subfigures(len(station_order), 1, hspace=0.06)

for s, (station_name, subfig) in enumerate(zip(station_order, subfigs)):
    df = snotel_dict[station_name]["df"]
    obs = df[("SNOTEL", "Observed")].dropna()

    subfig.suptitle(f"{station_name} SNOTEL", fontweight='bold', fontsize=14)
    axs = subfig.subplots(2, len(MODEL_NAMES))

    for col, model in enumerate(MODEL_NAMES):
        ax_ts, ax_sc = axs[0, col], axs[1, col]
        row_ts, row_sc = 2*s, 2*s + 1   # rows in the overall panel-label grid

        # --- Time series ---
        ax_ts.plot(obs.index, obs.values, color='k', linewidth=lw)
        for run, color in run_styles:
            ax_ts.plot(df.index, df[(model, run)], color=color, linewidth=lw)

        # Model names only on the very top row
        if s == 0:
            ax_ts.set_title(MODEL_NAMES_DISPLAY[model])#, fontweight='bold')

        # Panel label
        add_panel_label(ax_ts, panel_labels[row_ts, col])
        ax_ts.xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[4, 10], bymonthday=1))
        ax_ts.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%Y-%m"))
        ax_ts.tick_params(axis='x', rotation=45, labelsize=9)
        ax_ts.set_ylim(lim)
        ax_ts.set_yticks(sd_ticks)
        if col == 0:
            ax_ts.set_ylabel("Snow Depth (m)")

        # --- SNOTEL vs. modeled, every date where both exist ---
        for run, color in run_styles:
            pair = pd.concat([obs, df[(model, run)]], axis=1, keys=["obs", "mod"]).dropna()
            ax_sc.scatter(pair["obs"], pair["mod"], s=ms, alpha=alpha, color=color, edgecolor='none')

        ax_sc.plot(lim, lim, color='k', linestyle='--', linewidth=lw)
        add_panel_label(ax_sc, panel_labels[row_sc, col])
        ax_sc.set_xlim(lim)
        ax_sc.set_ylim(lim)
        ax_sc.set_xticks(sd_ticks)
        ax_sc.set_yticks(sd_ticks)
        ax_sc.set_xlabel("SNOTEL Snow Depth (m)")
        if col == 0:
            ax_sc.set_ylabel("Modeled Snow Depth (m)")

    subfig.align_ylabels(axs[:, 0])

# Separator line across the gap between the two station blocks
# fig.add_artist(Line2D([0.02, 0.98], [0.5, 0.5], transform=fig.transFigure, color='k', linewidth=0.8))

legend_handles = [
    Line2D([0], [0], color='k', linewidth=lw, label='SNOTEL'),
    Line2D([0], [0], color=base_run_color, linewidth=lw, label='Baseline Run'),
    Line2D([0], [0], color=assim_run_color, linewidth=lw, label='Assimilation Run'),
    Line2D([0], [0], color='k', linestyle='--', linewidth=lw, label='1:1 line'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.0))
plt.show()

fig_file = os.path.join(fig_dir, "SNOTEL_timeseries_scatter.png")
fig.savefig(fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", fig_file)

In [ ]:
lw = 1.5

def kge_score(obs, mod, snow_on_only=False):
    """KGE (Gupta et al. 2009, scores.continuous.kge, as in KGE.ipynb) over dates where both exist."""
    pair = pd.concat([obs, mod], axis=1, keys=["obs", "mod"]).dropna()
    if snow_on_only:
        pair = pair[(pair["obs"] > 0) | (pair["mod"] > 0)]
    return float(scores.continuous.kge(
        xr.DataArray(pair["mod"].values, dims="time"),
        xr.DataArray(pair["obs"].values, dims="time"),
        ))

# --- KGE for every station / model / run ---
kge_df = pd.DataFrame([
    dict(station=st, model=m, run=run,
         kge=kge_score(snotel_dict[st]["df"][("SNOTEL", "Observed")].dropna(), snotel_dict[st]["df"][(m, run)]))
    for st in station_order for m in MODEL_NAMES for run, _ in run_styles
])
display(kge_df.pivot_table(index=["station", "model"], columns="run", values="kge").round(2))

# --- Layout: n_st x n_mod time series + one KGE column on the right ---
n_st, n_mod = len(station_order), len(MODEL_NAMES)
lim_ts = (0, 3)
run_offsets = {run: off for (run, _), off in zip(run_styles, (-0.12, 0.12))}

fig = plt.figure(figsize=(14, 6), layout='constrained')
gs = fig.add_gridspec(n_st, n_mod + 1, width_ratios=[1] * n_mod + [0.6])

ts_axs = np.empty((n_st, n_mod), dtype=object)
for s in range(n_st):
    for col in range(n_mod):
        share = ts_axs[0, 0] if (s, col) != (0, 0) else None
        ts_axs[s, col] = fig.add_subplot(gs[s, col], sharex=share, sharey=share)
kge_axs = [fig.add_subplot(gs[s, -1]) for s in range(n_st)]


def plot_kge_points(ax, x, sub):
    """Baseline and assimilation KGE at x, joined by a thin line to show the change."""
    vals = sub.set_index("run")["kge"]
    xs = [x + run_offsets[run] for run, _ in run_styles]
    ax.plot(xs, [vals[run] for run, _ in run_styles], color='0.6', linewidth=0.8, zorder=1)
    for run, color in run_styles:
        ax.scatter(x + run_offsets[run], vals[run], s=40, color=color, edgecolor='k', linewidth=0.4, zorder=2)


# --- Time series ---
for s, station_name in enumerate(station_order):
    df = snotel_dict[station_name]["df"]
    obs = df[("SNOTEL", "Observed")].dropna()

    for col, model in enumerate(MODEL_NAMES):
        ax = ts_axs[s, col]
        ax.plot(obs.index, obs.values, color='k', linewidth=lw)
        for run, color in run_styles:
            ax.plot(df.index, df[(model, run)], color=color, linewidth=lw)
        if s == 0:
            ax.set_title(MODEL_NAMES_DISPLAY[model])
        if col == 0:
            ax.set_ylabel(f"{station_name}\nSnow Depth (m)")
        else:
            ax.tick_params(labelleft=False)
        if s < n_st - 1:
            ax.tick_params(labelbottom=False)
        ax.set_xlim(np.datetime64("2022-09-15"), np.datetime64("2025-06-15"))
        ax.set_ylim(lim_ts)
        ax.xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[4, 10], bymonthday=1))
        ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%Y-%m"))
        ax.tick_params(axis='x', rotation=45, labelsize=9)
        ax.grid(**GRID_STYLE)

# --- Right column: KGE per model, one panel per station ---
for s, (station_name, ax) in enumerate(zip(station_order, kge_axs)):
    for x, model in enumerate(MODEL_NAMES):
        plot_kge_points(ax, x, kge_df[(kge_df["station"] == station_name) & (kge_df["model"] == model)])
    ax.set_xticks(range(n_mod))
    ax.set_xticks(np.arange(0.5, n_mod - 0.5), minor=True)   # halfway between models
    ax.tick_params(axis='x', which='minor', length=0)        # separator lines only, no tick marks
    ax.grid(axis='x', which='minor', **GRID_STYLE)
    ax.set_xlim(-0.6, n_mod - 0.4)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', which='major', **GRID_STYLE)
    ax.yaxis.tick_right()
    ax.yaxis.set_label_position("right")
    ax.set_ylabel("KGE")
    if s < n_st - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xticklabels([MODEL_NAMES_DISPLAY[m] for m in MODEL_NAMES], rotation=45, ha='center', fontsize=9)

# Panel labels
ordered_axes = [a for s in range(n_st) for a in list(ts_axs[s]) + [kge_axs[s]]]
for label, ax in zip(string.ascii_lowercase, ordered_axes):
    add_panel_label(ax, label)

legend_handles = [
    Line2D([0], [0], color='k', linewidth=lw, label='SNOTEL'),
    Line2D([0], [0], color=base_run_color, linewidth=lw, marker='o', markeredgecolor='k',
           markeredgewidth=0.4, label='Baseline Run'),
    Line2D([0], [0], color=assim_run_color, linewidth=lw, marker='o', markeredgecolor='k',
           markeredgewidth=0.4, label='Assimilation Run'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=len(legend_handles), frameon=False,
           bbox_to_anchor=(0.5, 1.0))
plt.show()

fig_file = os.path.join(fig_dir, "SNOTEL_timeseries_kge.png")
fig.savefig(fig_file, dpi=300, bbox_inches='tight')
print("Figure saved to:", fig_file)